# High-precision news/social dataset

This notebook builds deterministic article/social-post pairs for the English section of Global Voices.

Precision rules:

- alignment uses only structured social URLs, HTTP redirects, and the article canonical URL;
- publisher social identities are pinned to a Bluesky DID or Mastodon actor URI;
- replies, boosts/reposts, quotes, comments, images, captions, related-content blocks, social embeds, and supported promotional blocks are excluded structurally;
- partner/republication notices in any retainable block, guest contributions, announcements, unwrapped promotions, terminal unwrapped cross-article teasers, and unsupported body structures cause whole-row rejection;
- title and body extraction requires exactly one publisher-specific DOM container;
- author extraction uses only publisher credit structures; multiple written authors retain DOM order and translators are excluded;
- license is fixed as CC BY 3.0 for Global Voices;
- no semantic matching, linguistic cleaning, case conversion, Unicode normalization, or whitespace collapsing is used;
- ambiguous or unsupported cases are rejected before a CSV row is stored.

Press **Interrupt Kernel** or `Ctrl+C` to stop safely. The SQLite checkpoint is committed per post. Running the notebook again resumes from the last uncompleted feed page. Connection and interrupted-response failures wait with bounded exponential backoff; repeated server or identity-verification failures pause the affected source without losing its cursor.

## 1. Install dependencies

Run this cell once for the current kernel, then restart the kernel if Jupyter requests it.

In [ ]:
%pip install -q -r requirements.txt

## 2. Configure the run

`selected_agencies` accepts `"global_voices"`. `max_pages_per_source` limits work in one invocation; use `None` to continue until each currently available social history is exhausted. A later run continues a paused history automatically.

In [ ]:
from english_corpus_generator import PipelineConfig, run_pipeline

config = PipelineConfig(
    selected_agencies=("global_voices",),
    max_pages_per_source=None,
    verbose=True,
)

config

## 3. Run or resume

The CSV is exported atomically at normal completion and after an operator interrupt. Rows are written to the checkpoint only after social ownership, deterministic link alignment, article ownership, canonical URL, title, and body structure all pass.

In [ ]:
summary = run_pipeline(config)
summary

## 4. Verify the exact CSV schema and preview rows

The verification parses every row, checks the exact schema and non-empty fields, and keeps a three-row preview. It does not rewrite the output.

In [ ]:
import csv

from english_corpus_generator import CSV_COLUMNS

with config.output_csv.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle, strict=True)
    assert tuple(reader.fieldnames or ()) == CSV_COLUMNS
    preview = []
    row_count = 0
    for row in reader:
        assert None not in row
        assert all(row[column] for column in CSV_COLUMNS)
        row_count += 1
        if len(preview) < 3:
            preview.append(row)

print(f"CSV: {config.output_csv.resolve()}")
print(f"Columns: {list(CSV_COLUMNS)}")
print(f"Rows: {row_count}")
preview